In [3]:
import sys #Library that let u inspect Python Version
print('python: ',sys.version) # Which pyhthon version is running 
print('location: ',sys.executable) # Where python is installed

python:  3.13.2 (tags/v3.13.2:4f8bb39, Feb  4 2025, 15:23:48) [MSC v.1942 64 bit (AMD64)]
location:  d:\final_chatbot_for_summarize_new\.venv\Scripts\python.exe


In [4]:
import os # read api from .env
import requests # code can talk to internet
from dotenv import load_dotenv #load secret key from venv
from newspaper import Article # it automatically scrapes the full article text for you

In [5]:
load_dotenv() #open .env file do this once 

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY') #Reads one specific key from it
NEWS_API_KEY = os.getenv('NEWS_API_KEY') #Reads another specific key from it

if not ANTHROPIC_API_KEY:
    print("Anthropic key not found — check your .env file!")
else:
    print("Anthropic key loaded!")
    print('start with: ', ANTHROPIC_API_KEY[:10] + '**********')
if not NEWS_API_KEY:

    print("NewsAPI key not found — check your .env file")
else:
    print("NewsAPI key loaded!")
    print("   Starts with:", NEWS_API_KEY[:10] + "****")

Anthropic key loaded!
start with:  sk-ant-api**********
NewsAPI key loaded!
   Starts with: 09d1213c4a****


In [6]:
from anthropic import Anthropic

def fetch_news(ticker : str , num_articles : int = 5) -> list :
    """
    Fetch top news articles for a given stock ticker.
    
    Args:
        ticker      : stock symbol e.g. "NVDA"
        num_articles: how many articles to fetch (default 5)
    
    Returns:
        list of dicts containing article title, url, and description
    """
    url = (
        f"https://newsapi.org/v2/everything?"
        f"q={ticker}&"
        f"language=en&"
        f"sortBy=publishedAt&"
        f"pageSize={num_articles}&"
        f"apiKey={NEWS_API_KEY}" #format พวกนี้ไปดูใน doc เอา
     ) #แบ่งได้เพราะถ้าให้เขียนใน one line จะดูไม่ดี

    response = requests.get(url)
    data = response.json()

    articles = []
    for article in data['articles']:
        articles.append({
            'title' : article['title'],
            'url' : article['url'],
            'description' : article['description']
        })
    
    return articles

The Output
#TitleURLDescription13 Stocks Nvidia Owns That You Should Consider Toohttps://biztoc.com/x/e2d4861422c54773Nvidia is the largest company in the world...2NVIDIA Is Stalled at $200. Is the Next Move a Breakout?https://biztoc.com/x/4ccb8c361d657db4Nvidia fell 4% to $200.58 midday Tuesday...3Billionaire Hedge Fund Manager: AI Could Produce $10 Trillion Company

INPUT                          OUTPUT
──────────────────────         ──────────────────────────────
ticker = "NVDA"         →      [
num_articles = 5                   {"title": "...", "url": "...", "description": "..."},
                                   {"title": "...", "url": "...", "description": "..."},
                                   {"title": "...", "url": "...", "description": "..."},
                                   {"title": "...", "url": "...", "description": "..."},
                                   {"title": "...", "url": "...", "description": "..."},
                               ]

In [7]:
articles = fetch_news("NVDA" , num_articles = 5)

for i , article in enumerate(articles):
    print(f'article  {i+1}')
    print('Title : ', article['title'])
    print('URL : ', article['url'])
    print('Description : ', article['description'])

article  1
Title :  Google’s parent company, Alphabet, is being added to the Dow Jones stock index
URL :  http://9to5google.com/2026/06/23/google-alphabet-dow-jones-index/
Description :  Effective next week, Google’s parent company, Alphabet, is being added to the Dow Jones stock index. Starting on June 29,...
article  2
Title :  Nvidia Is Officially the Largest Stock in the World. Is the Artificial Intelligence (AI) Giant Still Cheap?
URL :  https://biztoc.com/x/03b7812ae30f63c3
Description :  Nvidia Is Officially the Largest Stock in the World. Is the Artificial Intelligence (AI) Giant Still Cheap?
Nvidia (NASDAQ: NVDA) is the largest company in the world, and by a large margin. Second-place Alphabet (NASDAQ: GOOG) (NASDAQ: GOOGL) sits at a $4.5 t…
article  3
Title :  3 Stocks Nvidia Owns That You Should Consider Too
URL :  https://biztoc.com/x/e2d4861422c54773
Description :  Nvidia (NASDAQ: NVDA) is the largest company in the world because it's highly exposed to the AI build-out and

What Is newspaper3k
It is a Python library built specifically to:
Visit a URL                    →  article.download()
Read the raw HTML              →  article.parse()
Extract only the article text  →  article.text
Remove ads, menus, footers     →  handled automatically

In [8]:
from newspaper import Article

#เอา url มาจาก fetch_news  -> อันที่ได้มาใส่ในนี้
def scrap_article(url : str) -> str :
    """
    Visit a URL and extract the full article text.
    
    Args:
        url : the article URL from fetch_news()
    
    Returns:
        full article text as a string
    """

    article = Article(url) 
    '''
    Goes to the URL on the internet and downloads the raw HTML of the page.
    This line creates an Article object from the newspaper3k library and tells it which URL to work with.
    '''
    article.download()
    # Opening a webpage in your browser — but instead of displaying it visually, it saves the raw code behind the page. 
    article.parse()
    # Reads through that raw HTML and extracts only the article text — removes everything else. <div class="article"> 
    
    return article.text

In [9]:
test_url = "https://finance.yahoo.com/technology/ai/articles/cerebras-shares-sink-on-earnings-debut-with-margins-below-ai-chip-rivals-205043191.html"
full_text = scrap_article(test_url)

print('Characters scraped', len(full_text))
print(full_text)

Characters scraped 2507
By Stephen Nellis and Juby Babu

Cerebras Systems (CBRS) in its debut report as a public company forecast full-year profit margins would drop below the first-quarter ‌figures and lag levels of chip companies including Nvidia , sending shares down over 14% in ‌premarket trading on Wednesday.

The chip designer, which raised $5.55 billion in its IPO last month, is focused on inference, the process ​by which AI systems respond to user queries, and has tied much of its growth to OpenAI, including a $20 billion multi-year deal under which the ChatGPT creator will deploy 750 megawatts of Cerebras chips.

In its release in after-hours Tuesday, Cerebras forecast adjusted gross margins of 38% to 41% for full-year 2026, down from the 47% it ‌reported for the first quarter. While ⁠the projection is above analyst estimates of 29.58%, it is far below those of rivals such as Nvidia (NVDA), whose gross margins are in the mid-70% ⁠area, and Advanced Micro Devices, whose gross m

In [10]:
from anthropic import Anthropic

client = Anthropic()

def summarize_article(article_text : str , ticker :str = '' ):
    """
    Send a news article to Claude and get back a summary
    in both Thai and English.

    Args:
        article_text : the raw news article text
        ticker       : optional stock symbol e.g. "NVDA"

    Returns:
        dict with summary, ticker, and tokens used
    """
        

    system_prompt = """You are a financial analyst assistant specializing in investment
    news. You summarize news articles clearly and concisely. Always respond in this exact format:

    ENGLISH SUMMARY:
    [2-3 sentence summary in English]

    THAI SUMMARY:
    [2-3 sentence summary in Thai]

    IMPACT:
    [  One sentence on how this affects investors]"""

    # YOU WRITE THIS — the task you give Claude
    user_prompt = f"""Please summarize this financial news article.
    {"Focus on impact to " + ticker + " investors." if ticker else ""}

    Article:
    {article_text}"""

    response = client.messages.create( 
        model = "claude-haiku-4-5-20251001",
        max_tokens = 2024 ,
        system = system_prompt ,
        messages = [{'role' : 'user' , 'content' : user_prompt }] # This is list bcs chat have multiple sentence ijn conversation
        )
    raw_text = response.content[0].text
    return {
        'raw' : raw_text ,
        'ticker': ticker ,
        'token_use' : response.usage.input_tokens + response.usage.output_tokens
    }


def run_pipeline(ticker : str = '' , num_articles : int =3) -> str:
    """
    Full pipeline — fetch, scrape, and summarize news for a ticker.

    Args:
        ticker       : stock symbol e.g. "NVDA"
        num_articles : how many articles to process

    Returns:
        list of summaries
    """
    print(f'fetching {num_articles} articles for {ticker} ')
    articles = fetch_news("NVDA" , num_articles)     

    result = []
    for i , article  in enumerate(articles):
        print(f'article {i+1} of {num_articles}......')
        full_text = scrap_article(article['url'])
        summary = summarize_article(full_text, ticker)

        result.append({
            'title': article['title'],
            'summary' : summary['raw'] ,
            'tokens_used' : summary['tokens_used'] ,
        })
    return result

print("Pipeline defined successfully.")
     

Pipeline defined successfully.


In [11]:
result = run_pipeline("NVDA"  , 5)

fetching 5 articles for NVDA 
article 1 of 5......


KeyError: 'tokens_used'